# LocalCodeGraph: frozen repository holdout

Run after the v2 policy is finalized. Twelve pinned test tasks from four repositories outside the analyzed dev set. Do not tune the policy, choose retries or filter failures after seeing these results. This is a small holdout with possible pretraining contamination, not a competition score.


In [ ]:
import json, os, pathlib, shutil, subprocess, sys
ROOT = pathlib.Path("/content/LocalCodeGraph")
VENV = pathlib.Path("/content/lcg-t4-env")
MODEL_DIR = pathlib.Path("/content/gemma-4-e2b")
INPUTS = pathlib.Path("/content/lcg-t4-holdout-inputs")
OUTPUT = pathlib.Path("/content/lcg-t4-holdout-run")
MODEL = "google/gemma-4-E2B-it"
RESUME = False  # True only to continue this exact run; completed/errors are never retried.
gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True)
print(gpu)
if not any(int(line.rsplit(",", 1)[1]) >= 14000 for line in gpu.strip().splitlines()):
    raise RuntimeError("Select a T4 GPU runtime with approximately 15 GiB usable VRAM.")
if shutil.disk_usage("/content").free < 30 * 1024**3 and not MODEL_DIR.exists():
    raise RuntimeError("Need 30 GiB free disk for initial installation and base-model download.")
if OUTPUT.exists() and not RESUME:
    raise RuntimeError("Output already exists. Preserve it or set RESUME=True for this exact run.")

SOURCES = pathlib.Path("/content/lcg-t4-holdout-source-snapshots")


## Install without replacing Colab's CUDA/PyTorch stack

A virtual environment reuses the installed Colab packages. New inference dependencies stay in that environment. The exact resolved versions are saved with the results. No GPU package is installed into the notebook kernel.


In [ ]:
def checked(command):
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout, flush=True)
    if result.returncode:
        print(result.stderr, flush=True)
        raise RuntimeError("Setup command failed; complete diagnostics are printed above.")
    return result

if not ROOT.exists():
    checked(["git", "clone", "https://github.com/eluan216/LocalCodeGraph.git", str(ROOT)])
else:
    # Bring an earlier clone forward without resetting local edits.
    checked(["git", "-C", str(ROOT), "pull", "--ff-only"])
REPO_COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
checked([sys.executable, str(ROOT / "colab/bootstrap_env.py"), "--destination", str(VENV)])
PYTHON = str(VENV / "bin/python")
checked([PYTHON, "-m", "pip", "install", "--upgrade",
    "-e", str(ROOT), "transformers==5.22.2", "accelerate==1.15.0", "bitsandbytes==0.50.2", "huggingface_hub",
    "pyarrow==26.0.1", "unidiff==1.0.1"])
print("Toolkit commit:", REPO_COMMIT)


## Prepare paired inputs

Only issue text and pre-fix context enter the generator. The frozen dataset has 23 public development tasks. Existing inputs are checksum-validated by the controller; do not change them between arms.


In [ ]:
if not INPUTS.exists():
    subprocess.run([PYTHON, str(ROOT / "benchmarks/prepare_repair_inputs.py"),
                    "--output-dir", str(INPUTS), "--holdout-manifest",
                    str(ROOT / "benchmarks/repair-holdout-v1.json")], check=True)
manifest = json.loads((INPUTS / "manifest.json").read_text())
assert manifest["tasks"] == 12
print("Input SHA256:", manifest["inputs_sha256"])

subprocess.run([PYTHON, str(ROOT / "benchmarks/prepare_edit_snapshots.py"),
    "--inputs", str(INPUTS / "inputs.jsonl"), "--manifest", str(INPUTS / "manifest.json"),
    "--output-dir", str(SOURCES)], check=True)


## Download E2B at one immutable revision

Quantization happens during model loading. The download contains original base weights, so the disk requirement is larger than final GPU storage. If authentication is needed, use Colab Secrets named HF_TOKEN; do not paste a token into code. Resume reuses the previously recorded revision.


In [ ]:
download_env = os.environ.copy()
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        download_env["HF_TOKEN"] = token
except Exception:
    pass
download_code = r'''
import json, sys
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
model, destination = sys.argv[1:]
marker = Path(destination) / "lcg-revision.json"
revision = json.loads(marker.read_text())["revision"] if marker.exists() else "4adf15748de9e1a0097b6274cee9c15836c41c37"
snapshot_download(model, revision=revision, local_dir=destination,
    allow_patterns=["*.json", "*.jinja", "*.safetensors", "*.model", "*.txt", "README.md", "LICENSE*"])
marker.write_text(json.dumps({"model": model, "revision": revision}))
print("Model revision:", revision)
'''
subprocess.run([PYTHON, "-c", download_code, MODEL, str(MODEL_DIR)],
               env=download_env, check=True)
download_env.pop("HF_TOKEN", None)
MODEL_REVISION = json.loads((MODEL_DIR / "lcg-revision.json").read_text())["revision"]


## Generate validated edits

Both arms retain the v1 model revision, quantization and token caps. Each gets at most two generations: an initial JSON edit proposal and one retry after validation failure. Retry feedback uses the same input cap and can displace source context. All responses, errors, timings and prompt hashes are saved. Runtime failures are not retried. Test outcomes are never used as feedback.


In [ ]:
command = [PYTHON, "-m", "localcodegraph.repair_t4_v2",
    "--inputs", str(INPUTS / "inputs.jsonl"),
    "--manifest", str(INPUTS / "manifest.json"),
    "--model-dir", str(MODEL_DIR), "--model-revision", MODEL_REVISION,
    "--output-dir", str(OUTPUT), "--source-dir", str(SOURCES)]
if RESUME or OUTPUT.exists():
    command.append("--resume")
inference_env = os.environ.copy()
inference_env.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
log_path = pathlib.Path("/content/lcg-v2-inference.log")
with log_path.open("w") as log:
    process = subprocess.Popen(command, env=inference_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    return_code = process.wait()
if return_code:
    print(log_path.read_text()[-12000:])
    raise RuntimeError("Inference failed; full output saved to " + str(log_path))
print("Generation finished. Patch statuses are not test-based repair scores.")


## Save outputs—even after an interrupted generation cell

Run this cell whenever you want to save partial work. Attempts are exported after every completion. A Colab VM deletion can erase files, so download before leaving. Model weights are not included in the ZIP. The exact gold-free inputs and upstream notices are included.


In [ ]:
if not OUTPUT.exists():
    raise RuntimeError("No run output exists yet.")
audit_result = subprocess.run([PYTHON, str(ROOT / "benchmarks/inspect_structured_run.py"),
    "--output-dir", str(OUTPUT), "--inputs", str(INPUTS / "inputs.jsonl"),
    "--manifest", str(INPUTS / "manifest.json"), "--source-dir", str(SOURCES)])
if audit_result.returncode:
    print("CPU audit failed; preserving the ZIP for diagnosis. Do not treat exports as validated.")
(OUTPUT / "environment.txt").write_text(
    subprocess.check_output([PYTHON, "-m", "pip", "freeze"], text=True))
(OUTPUT / "hardware.txt").write_text(subprocess.check_output(["nvidia-smi"], text=True))
(OUTPUT / "toolkit-commit.txt").write_text(REPO_COMMIT + "\n")
shutil.copytree(INPUTS, OUTPUT / "gold-free-inputs", dirs_exist_ok=True)
shutil.copy2(SOURCES / "manifest.json", OUTPUT / "source-snapshot-checksums.json")
archive = shutil.make_archive("/content/localcodegraph-t4-e2b-holdout", "zip", OUTPUT)
from google.colab import files
files.download(archive)


## Grade all planned tasks

Use the pinned dataset test split and frozen 12-task manifest. Validate fixed and unfixed controls per project; report every failure and missing outcome. No holdout results exist until this notebook and separate test grading have run.
